# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường

Mã thật nằm trong `lab/` (mỗi bước của GUIDE là một module); notebook này gọi lại các module đó theo đúng thứ tự
GUIDE và in số liệu thật.

- **GPU:** mọi bước huấn luyện/suy luận chạy trên **Kaggle T4 x2** (2 thí nghiệm song song, mỗi GPU một thí nghiệm)
  bằng `kaggle_jobs/build.py` → log trong `report/logs/`. Các ô đó để `# (đã chạy trên Kaggle)` và không có output.
- **CPU:** các ô còn lại (dữ liệu, nhãn, chấm điểm, phân tích, test reward) được chạy thật khi dựng notebook
  (`python -m lab.build_notebook`).

## 0. Môi trường

In [ ]:
# (đã chạy trên Kaggle) — runner cài đúng các bản đã ghim rồi gỡ torchao cũ của image Kaggle
# !pip -q install transformers==5.19.0 peft==0.21.2 trl==1.9.2 accelerate==1.15.0 bitsandbytes==0.50.2
# !pip uninstall -y torchao
# !python -m pytest -q

In [1]:
import json, os, subprocess, sys
sys.path.insert(0, '.')
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests', 'lab/tests'], capture_output=True, text=True,
                   encoding='utf-8', errors='replace')
print(r.stdout.strip().splitlines()[-1])

2 failed, 78 passed, 1 skipped in 0.90s


## 1. Dữ liệu: split theo phiên quay

In [2]:
# !python scripts/prepare_data.py --kaggle      (trên Kaggle: --source /kaggle/input/...)
from lab.data_check import stats
st, overlap = stats()
print(json.dumps(st, indent=1, ensure_ascii=False))
print('phiên nằm ở 2 split:', overlap)
print('dataset_version:', json.load(open('data/splits/meta.json'))['dataset_version'])

{
 "train": {
  "images": 1422,
  "boxes": {
   "pothole": 1074,
   "crack": 1750,
   "manhole": 691
  },
  "empty_images": 0,
  "sessions": 13,
  "median_box_area_pct": 0.644
 },
 "val": {
  "images": 276,
  "boxes": {
   "pothole": 66,
   "crack": 470,
   "manhole": 103
  },
  "empty_images": 0,
  "sessions": 2,
  "median_box_area_pct": 1.009
 },
 "test": {
  "images": 311,
  "boxes": {
   "pothole": 121,
   "crack": 299,
   "manhole": 163
  },
  "empty_images": 0,
  "sessions": 3,
  "median_box_area_pct": 1.063
 }
}
phiên nằm ở 2 split: {'train&val': [], 'train&test': [], 'val&test': []}
dataset_version: d10-21e4070301


![6 ảnh train kèm box gốc](report/figures/data_gt_samples.jpg)

**Chia ngẫu nhiên thì sao?** Baseline *không học gì*: chép box của ảnh train giống nhất (ảnh xám 64×36, L2).
Split theo phiên: láng giềng không bao giờ cùng phiên → F1 ≈ 0. Split ngẫu nhiên: láng giềng là frame liền kề → F1 cao giả.

In [3]:
# python -m lab.leakage --out report/analysis/leakage.json
lk = json.load(open('report/analysis/leakage.json', encoding='utf-8'))
s = lk['session_split']
print(f"split theo phiên : F1={s['f1']:.3f}  láng giềng cùng phiên={s['nn_same_session_frac']:.0%}  gần trùng={s['near_duplicate_frac']:.0%}")
for r in lk['random_split']:
    print(f"{r['split']:<20}: F1={r['f1']:.3f}  láng giềng cùng phiên={r['nn_same_session_frac']:.0%}  gần trùng={r['near_duplicate_frac']:.0%}")

split theo phiên : F1=0.003  láng giềng cùng phiên=0%  gần trùng=1%
random seed=0       : F1=0.105  láng giềng cùng phiên=85%  gần trùng=35%
random seed=1       : F1=0.077  láng giềng cùng phiên=84%  gần trùng=31%
random seed=2       : F1=0.082  láng giềng cùng phiên=84%  gần trùng=31%
random seed=3       : F1=0.092  láng giềng cùng phiên=85%  gần trùng=36%
random seed=4       : F1=0.102  láng giềng cùng phiên=85%  gần trùng=38%


## 2. Nhãn YOLO → JSON thang 0–1000

In [4]:
from lab.common import build_prompt, objects_to_answer, yolo_to_box, box_to_pixels, load_split
e = load_split('train')[0]
print(e['image_id'], e['objects'][:2])
print(objects_to_answer(e['objects']))
# Kiểm tra: quy ngược về pixel phải trùng box YOLO gốc (sai số chỉ do làm tròn số nguyên)
err, skipped = 0, 0
for split in ('train', 'val', 'test'):
    for x in load_split(split):
        for o in x['objects']:
            xc, yc, w, h = o['yolo']
            if w * h == 0 or not (0 <= xc - w/2 and xc + w/2 <= 1 and 0 <= yc - h/2 and yc + h/2 <= 1):
                skipped += 1  # box suy biến / tràn khung: bị kẹp có chủ đích
                continue
            px = box_to_pixels(yolo_to_box(o['label'], o['yolo'])['bbox_2d'], 'norm1000', 640, 360)
            ref = ((xc-w/2)*640, (yc-h/2)*360, (xc+w/2)*640, (yc+h/2)*360)
            err = max(err, max(abs(p - q) for p, q in zip(px, ref)))
print(f'sai số quy đổi lớn nhất: {err:.2f} px ({skipped} box suy biến/tràn khung được kẹp)')
print(build_prompt('norm1000'))

20250216_164325 [{'label': 'manhole', 'yolo': [0.466407, 0.552778, 0.298437, 0.144444]}, {'label': 'pothole', 'yolo': [0.604688, 0.556944, 0.043751, 0.113889]}]
[{"label":"crack","bbox_2d":[430,467,583,506]},{"label":"pothole","bbox_2d":[361,475,414,522]},{"label":"manhole","bbox_2d":[317,481,616,625]},{"label":"pothole","bbox_2d":[583,500,627,614]},{"label":"crack","bbox_2d":[580,508,655,614]},{"label":"pothole","bbox_2d":[259,517,311,572]},{"label":"crack","bbox_2d":[397,628,447,717]},{"label":"crack","bbox_2d":[339,683,398,808]},{"label":"crack","bbox_2d":[158,789,361,994]}]
sai số quy đổi lớn nhất: 0.32 px (64 box suy biến/tràn khung được kẹp)
Locate every road damage instance in this dashcam image. Classes: pothole, crack, manhole. Answer with ONLY a JSON list in the form [{"label": "pothole", "bbox_2d": [x1, y1, x2, y2]}], where x1, y1, x2, y2 are integers on a 0-1000 scale relative to the image width and height (0,0 is the top-left corner, 1000,1000 is the bottom-right corner). 

## 3. E0 zero-shot

In [ ]:
# (đã chạy trên Kaggle) — lab/common.py: llm_int8_skip_modules=["model.visual", "visual", "lm_head"]
# !python -m lab.infer --out work/preds/e0.jsonl
# !python scripts/score.py --pred work/preds/e0.jsonl --name E0 --out report/evidence/e0.json

In [5]:
from scripts.score import parse_output
e0 = json.load(open('report/evidence/e0.json'))
print('parse_valid_rate =', round(e0['parse_valid_rate'], 3), '| F1 =', round(e0['micro']['f1'], 3))
raws = [json.loads(l) for l in open('report/preds/e0.jsonl', encoding='utf-8')]
bad = [r for r in raws if parse_output(r['raw']) is None]
print(f'JSON hỏng: {len(bad)} ảnh, trong đó {sum(len(r["raw"]) >= 400 for r in bad)} bị cắt ở 256 token vì lặp box')
for r in raws[:3] + bad[:2]:
    print('-', r['image_id'], repr(r['raw'][:160]))

parse_valid_rate = 0.907 | F1 = 0.101
JSON hỏng: 29 ảnh, trong đó 25 bị cắt ở 256 token vì lặp box
- vlcsnap-00200 '[{"label": "manhole", "bbox_2d": [770, 640, 843, 684]}]'
- vlcsnap-00201 '[]'
- vlcsnap-00202 '[{"label": "manhole", "bbox_2d": [309, 686, 399, 719]}]'
- vlcsnap-2025-02-18-23h08m10s837 '[{"label": "crack", "bbox_2d": [295, 478, 357, 500]}, {"label": "crack", "bbox_2d": [295, 478, 357, 500)}]'
- vlcsnap-2025-02-18-23h08m18s609 '[{"label": "crack", "bbox_2d": [407, 420, 475, 470]}, {"label": "crack", "bbox_2d": [475, 420, 530, 470]}, {"label": "crack", "bbox_2d": [530, 420, 585, 470]}, '


## 4. Ví dụ huấn luyện và mask loss

In [6]:
# (đã chạy trên Kaggle: python -m lab.inspect_model) — mã: lab/data.py::encode_example, make_collate
ins = json.load(open('report/analysis/inspect_model.json', encoding='utf-8'))
enc = ins['encoding']
print('image_grid_thw =', enc['image_grid_thw'], '→ token ảnh =', enc['n_image_tokens'])
print('độ dài chuỗi =', enc['seq_len'], '| token có loss =', enc['n_answer_tokens'])
print('giải mã labels != -100:', enc['decoded_labels'][:120], '...', enc['decoded_labels'][-30:])

image_grid_thw = [1, 22, 40] → token ảnh = 220
độ dài chuỗi = 605 | token có loss = 249
giải mã labels != -100: [{"label":"crack","bbox_2d":[430,467,583,506]},{"label":"pothole","bbox_2d":[361,475,414,522]},{"label":"manhole","bbox_ ... :[158,789,361,994]}]<|im_end|>


**Đối chứng không mask** (`python -m lab.train --setup E1 --no-mask-prompt`, đã chạy trên Kaggle): cùng E1, chỉ đổi mask.

In [7]:
for k in ('e4_coord_pixel', 'nomask_e1'):
    log = [json.loads(l) for l in open(f'report/runs/{k}/train_log.jsonl')]
    t, s = sum(x['tokens'] for x in log), sum(x['supervised_tokens'] for x in log)
    print(f"{k:<15} token/mẫu={t/len(log)/8:.1f}  token có loss/mẫu={s/len(log)/8:.1f} ({s/t:.1%})  "
          f"loss {sum(x['loss'] for x in log[:10])/10:.2f} → {sum(x['loss'] for x in log[-10:])/10:.2f}")
for k in ('report/evidence/e1.json', 'report/extra/nomask_e1.json'):
    d = json.load(open(k))
    print(f"{d['name']:<10} F1={d['micro']['f1']:.3f} CI95={[round(x, 3) for x in d['f1_ci95']]} "
          f"P={d['micro']['precision']:.3f} R={d['micro']['recall']:.3f} FP/ảnh={d['fp_per_image']:.2f}")

e4_coord_pixel  token/mẫu=427.3  token có loss/mẫu=70.3 (16.5%)  loss 1.25 → 0.53
nomask_e1       token/mẫu=426.6  token có loss/mẫu=426.6 (100.0%)  loss 11.56 → 4.14
E1         F1=0.268 CI95=[0.232, 0.309] P=0.276 R=0.261 FP/ảnh=1.28
E1-nomask  F1=0.255 CI95=[0.219, 0.292] P=0.240 R=0.273 FP/ảnh=1.62


## 5. Setup LoRA / đóng băng

In [8]:
print(json.dumps(ins['block_params'], indent=1))
print('số module LoRA:', ins['lora_modules'], '| dtype trên GPU:', ins['gpu_dtypes'])
for k in ('e1', 'e2', 'e3'):
    r = json.load(open(f'report/runs/{k}/run.json'))
    print(k.upper(), {x: r[x] for x in ('n_lora_modules', 'trainable_params', 'lora_params', 'projector_params')})

{
 "vit (patch_embed+pos_embed+blocks)": 306242560,
 "projector.merger": 27271680,
 "projector.deepstack_merger_list": 81833472,
 "llm.embed_tokens": 388956160,
 "llm.layers+norm": 3633511936,
 "total": 4437815808
}
số module LoRA: {'E1': 252, 'E2': 252, 'E3': 348} | dtype trên GPU: {'visual.blocks.0.attn.qkv': 'torch.float16', 'visual.merger.linear_fc1': 'torch.float16', 'lm_head': 'torch.float16', 'llm q_proj': 'Linear4bit'}
E1 {'n_lora_modules': 252, 'trainable_params': 33030144, 'lora_params': 33030144, 'projector_params': 0}
E2 {'n_lora_modules': 252, 'trainable_params': 142135296, 'lora_params': 33030144, 'projector_params': 109105152}
E3 {'n_lora_modules': 348, 'trainable_params': 148426752, 'lora_params': 39321600, 'projector_params': 109105152}


## 6. Huấn luyện (300 bước, batch 2 × tích luỹ 4, lr 2e-4, cosine, fp16 + GradScaler)

In [ ]:
# (đã chạy trên Kaggle, GPU0: E1 rồi E2; GPU1: E3 rồi E0)
# !python -m lab.train --setup E1 --out work/runs/e1
# !python -m lab.train --setup E2 --out work/runs/e2      # + projector.pt lưu riêng
# !python -m lab.train --setup E3 --out work/runs/e3

In [9]:
for k in ('e1', 'e2', 'e3'):
    r = json.load(open(f'report/runs/{k}/run.json'))
    log = [json.loads(l) for l in open(f'report/runs/{k}/train_log.jsonl')]
    print(f"{k.upper()}: loss {sum(x['loss'] for x in log[:10])/10:.3f} → {sum(x['loss'] for x in log[-10:])/10:.3f} | "
          f"VRAM đỉnh {r['peak_vram_gb']} GB | {r['train_minutes']} phút | {r['gpu']}")

E1: loss 1.272 → 0.590 | VRAM đỉnh 6.49 GB | 42.97 phút | Tesla T4
E2: loss 1.227 → 0.546 | VRAM đỉnh 8.18 GB | 43.88 phút | Tesla T4
E3: loss 1.218 → 0.503 | VRAM đỉnh 8.35 GB | 51.78 phút | Tesla T4


![loss](report/figures/loss_e1_e3.png)

## 7. Suy luận và chấm E1–E3

In [ ]:
# (đã chạy trên Kaggle) nạp 4-bit → adapter LoRA → projector.pt (E2/E3)
# !python -m lab.infer --run-dir work/runs/e3 --out work/preds/e3.jsonl
# !python scripts/score.py --pred work/preds/e3.jsonl --name E3 --out report/evidence/e3.json $(python -m lab.meta_args work/runs/e3)

In [10]:
print(open('report/analysis/table.md', encoding='utf-8').read())

| Mã | Setup | Tham số huấn luyện | VRAM đỉnh (GB) | Phút | F1 | CI95 | FP/ảnh | F1 pothole | F1 crack | F1 manhole | P | R | valid | mean IoU |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| E0 | zero-shot | 0 | – | – | 0.101 | [0.074, 0.131] | 0.86 | 0.121 | 0.033 | 0.222 | 0.144 | 0.077 | 0.907 | 0.614 |
| E1 | LoRA LLM | 33,030,144 | 6.49 | 42.97 | 0.268 | [0.232, 0.309] | 1.28 | 0.113 | 0.217 | 0.529 | 0.276 | 0.261 | 0.994 | 0.681 |
| E2 | + projector | 142,135,296 | 8.18 | 43.88 | 0.313 | [0.276, 0.353] | 1.20 | 0.167 | 0.239 | 0.589 | 0.322 | 0.304 | 1.000 | 0.674 |
| E3 | + LoRA ViT | 148,426,752 | 8.35 | 51.78 | 0.429 | [0.389, 0.469] | 0.87 | 0.316 | 0.337 | 0.702 | 0.462 | 0.400 | 1.000 | 0.700 |
| E4 | E4 coord=norm1000 | 148,426,752 | 8.35 | 51.78 | 0.429 | [0.389, 0.469] | 0.87 | 0.316 | 0.337 | 0.702 | 0.462 | 0.400 | 1.000 | 0.700 |
| E4 | E4 coord=pixel | 148,426,752 | 8.35 | 55.91 | 0.363 | [0.323, 0.407] | 1.08 | 0.241 | 0.288 | 0.608 | 0.378 | 0.35

![F1 kèm CI95](report/figures/f1_ci95.png)

In [11]:
an = json.load(open('report/analysis/analysis.json', encoding='utf-8'))
for k, v in an['paired'].items():
    print(f"{k:<22} ΔF1={v['diff']:+.3f}  CI95 cặp=[{v['ci95'][0]:+.3f}, {v['ci95'][1]:+.3f}]")

e1-e0                  ΔF1=+0.168  CI95 cặp=[+0.130, +0.205]
e2-e1                  ΔF1=+0.044  CI95 cặp=[+0.007, +0.081]
e3-e2                  ΔF1=+0.116  CI95 cặp=[+0.076, +0.156]
e3-e1                  ΔF1=+0.160  CI95 cặp=[+0.120, +0.203]
e5-e3                  ΔF1=+0.019  CI95 cặp=[-0.002, +0.039]
e4_coord_pixel-e3      ΔF1=-0.065  CI95 cặp=[-0.102, -0.029]
nomask_e1-e1           ΔF1=-0.013  CI95 cặp=[-0.045, +0.015]


## 8. E4: ablation thang toạ độ (norm1000 vs pixel) trên setup tốt nhất (E3)

In [ ]:
# (đã chạy trên Kaggle)
# !python -m lab.train --setup E3 --coord pixel --out work/runs/e4_coord_pixel
# !python -m lab.infer --run-dir work/runs/e4_coord_pixel --coord pixel --out work/preds/e4_coord_pixel.jsonl
# !python scripts/score.py --pred work/preds/e4_coord_pixel.jsonl --name E4 --coord pixel --out report/evidence/e4_coord_pixel.json ...

In [12]:
for k in ('e4_coord_norm1000', 'e4_coord_pixel'):
    p = f'report/evidence/{k}.json'
    if os.path.exists(p):
        d = json.load(open(p))
        print(k, '| coord =', d['coord'], '| F1 =', round(d['micro']['f1'], 3), '| CI95 =', [round(x, 3) for x in d['f1_ci95']])
v = an['paired']['e4_coord_pixel-e3']
print(f"pixel − norm1000: ΔF1={v['diff']:+.3f}  CI95 cặp=[{v['ci95'][0]:+.3f}, {v['ci95'][1]:+.3f}]")
print('chấm sai thang:', json.dumps(an['wrong_scale'], indent=1))

e4_coord_norm1000 | coord = norm1000 | F1 = 0.429 | CI95 = [0.389, 0.469]
e4_coord_pixel | coord = pixel | F1 = 0.363 | CI95 = [0.323, 0.407]
pixel − norm1000: ΔF1=-0.065  CI95 cặp=[-0.102, -0.029]
chấm sai thang: {
 "e0": {
  "scored_as": "pixel",
  "f1": 0.0,
  "mean_iou": 0.0,
  "right_f1": 0.10055865921787709
 },
 "e3": {
  "scored_as": "pixel",
  "f1": 0.0,
  "mean_iou": 0.0,
  "right_f1": 0.42870285188592455
 },
 "e4_coord_pixel": {
  "scored_as": "norm1000",
  "f1": 0.0,
  "mean_iou": 0.0,
  "right_f1": 0.3633125556544969
 }
}


## 9. E5 (bonus): GRPO với reward IoU

In [13]:
from lab.reward import compute_reward
GT = [{'label': 'pothole', 'bbox': (100, 100, 200, 200)}]
for name, out in [('hoàn hảo', '[{"label":"pothole","bbox_2d":[100,100,200,200]}]'),
                  ('JSON hỏng', '[{"label":"pothole","bbox_2d":[100,'),
                  ('sai lớp', '[{"label":"crack","bbox_2d":[100,100,200,200]}]'),
                  ('thừa box', '[{"label":"pothole","bbox_2d":[100,100,200,200]},{"label":"pothole","bbox_2d":[800,800,900,900]}]'),
                  ('box khổng lồ', '[{"label":"pothole","bbox_2d":[0,0,1000,1000]}]'),
                  ('trả []', '[]')]:
    print(f'{name:<12} reward={compute_reward(out, GT):.2f}')
print('cùng rỗng    reward=%.2f' % compute_reward('[]', []))
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'lab/tests'], capture_output=True, text=True,
                   encoding='utf-8', errors='replace')
print(r.stdout.strip().splitlines()[-1])

hoàn hảo     reward=1.00
JSON hỏng    reward=0.00
sai lớp      reward=0.20
thừa box     reward=0.60
box khổng lồ reward=0.20
trả []       reward=0.20
cùng rỗng    reward=1.00
12 passed in 0.03s


In [14]:
# (đã chạy trên Kaggle) !python -m lab.grpo --init-run work/runs/e3 --out work/runs/e5
p = 'report/runs/e5/run.json'
if os.path.exists(p):
    print(json.dumps({k: v for k, v in json.load(open(p)).items() if k != 'gpu'}, indent=1))
    rs = [json.loads(l) for l in open('report/runs/e5/reward_stats.jsonl', encoding='utf-8')]
    q = max(1, len(rs) // 4)
    for name, part in (('1/4 đầu', rs[:q]), ('1/4 cuối', rs[-q:])):
        avg = lambda key: sum(x[key] for x in part) / len(part)
        print(f"{name}: reward={avg('reward_mean'):.3f} rỗng={avg('empty_frac'):.1%} box khổng lồ={avg('giant_box_frac'):.1%} "
              f"JSON hỏng={avg('invalid_frac'):.1%} box/ảnh={avg('pred_boxes_mean'):.2f} (GT {avg('gt_boxes_mean'):.2f})")
    e5 = json.load(open('report/evidence/e5.json'))
    v = an['paired']['e5-e3']
    print(f"E5: F1={e5['micro']['f1']:.3f} CI95={[round(x, 3) for x in e5['f1_ci95']]} P={e5['micro']['precision']:.3f} "
          f"R={e5['micro']['recall']:.3f} | E5 − E3: ΔF1={v['diff']:+.3f} CI95 cặp=[{v['ci95'][0]:+.3f}, {v['ci95'][1]:+.3f}]")

{
 "setup": "E5-GRPO",
 "init_run": "work/runs/e3",
 "trainable_params": 39321600,
 "peak_vram_gb": 7.47,
 "train_minutes": 46.29,
 "train_prompts": 1280,
 "max_steps": 60,
 "lr": 5e-06,
 "beta": 0.0,
 "num_generations": 4,
 "max_completion_length": 128,
 "batch_size": 8,
 "grad_accum": 2,
 "limit": 0,
 "seed": 42
}
1/4 đầu: reward=0.426 rỗng=0.0% box khổng lồ=0.0% JSON hỏng=3.8% box/ảnh=1.90 (GT 1.93)
1/4 cuối: reward=0.452 rỗng=0.0% box khổng lồ=6.7% JSON hỏng=1.2% box/ảnh=1.66 (GT 1.97)
E5: F1=0.448 CI95=[0.407, 0.488] P=0.509 R=0.400 | E5 − E3: ΔF1=+0.019 CI95 cặp=[-0.002, +0.039]


![GRPO](report/figures/grpo_reward.png)

## 10. Phân tích lỗi (câu 9) và ảnh minh hoạ

In [15]:
best = an['best_sft']
print('setup SFT tốt nhất:', best)
print('FP:', json.dumps(an['errors'][best]['fp']))
print('FN:', json.dumps(an['errors'][best]['fn']))
print('recall theo diện tích box:', json.dumps(an['recall_by_size'][best], ensure_ascii=False))
print('F1 theo phiên quay:', an['session_bootstrap'][best]['f1_per_session'])

setup SFT tốt nhất: e3
FP: {"pothole": {"nham_lop": 17, "sai_vi_tri": 11, "bia": 27}, "crack": {"nham_lop": 19, "sai_vi_tri": 96, "bia": 79}, "manhole": {"nham_lop": 4, "sai_vi_tri": 8, "bia": 10}, "other": {"nham_lop": 0, "sai_vi_tri": 0, "bia": 0}}
FN: {"pothole": {"nham_lop": 15, "sai_vi_tri": 11, "bo_sot": 62}, "crack": {"nham_lop": 4, "sai_vi_tri": 126, "bo_sot": 69}, "manhole": {"nham_lop": 13, "sai_vi_tri": 8, "bo_sot": 42}}
recall theo diện tích box: {"pothole": {"<0.25%": {"recall": 0.212, "n": 33}, "0.25-1.0%": {"recall": 0.295, "n": 61}, "1.0-4.0%": {"recall": 0.292, "n": 24}, ">=4.0%": {"recall": 0.333, "n": 3}}, "crack": {"<0.25%": {"recall": 0.0, "n": 7}, "0.25-1.0%": {"recall": 0.098, "n": 61}, "1.0-4.0%": {"recall": 0.351, "n": 168}, ">=4.0%": {"recall": 0.556, "n": 63}}, "manhole": {"<0.25%": {"recall": 0.154, "n": 13}, "0.25-1.0%": {"recall": 0.612, "n": 103}, "1.0-4.0%": {"recall": 0.745, "n": 47}, ">=4.0%": {"recall": null, "n": 0}}}
F1 theo phiên quay: {'num-4': 0.

Ảnh minh hoạ: `report/figures/e3_*.jpg` và `e0_*.jpg` (xanh lá: GT khớp, vàng: GT bỏ sót, xanh dương: dự đoán đúng, đỏ: dự đoán sai).